# 04 · Modelo de estancia (LOS) + SHAP
Dos versiones:
- **Al ingreso** (solo info disponible al llegar) → planeación.
- **Explicativo** (toda la visita) → entender factores.

Validación con **corte temporal**.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config, load, quality, flow, features, models

# Fuente de datos:
#   "db"     -> base de datos del reto (FortiClient VPN conectada + archivo .env)
#   "raw"    -> archivos exportados en data/raw
#   "sample" -> datos sintéticos de prueba (scripts/generar_datos_sinteticos.py)
SOURCE = "db"
DATA_DIR = ROOT / "data" / ("sample" if SOURCE == "sample" else "raw")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
if SOURCE == "db":
    from src import db
    db.test_connection()
    t = db.load_all()   # copia local en data/interim; db.load_all(refresh=True) para volver a descargar
else:
    t = load.load_all(DATA_DIR)

In [ ]:
df = features.build_visit_table(t)
df = df.dropna(subset=['los_h'])
print(df.shape)
df[['los_h','edad','tipo_visita','ocupacion_llegada','n_etapas']].describe(include='all').T

In [ ]:
tr_idx, te_idx, cut = models.temporal_split(df)
print('Corte temporal:', cut)

## Modelo al ingreso — regresión de log(LOS)

In [ ]:
X = models.prepare_X(df, features.FEATURES_INGRESO)
y = df['log_los']
reg = models.train_regressor(X.loc[tr_idx], y.loc[tr_idx], X.loc[te_idx], y.loc[te_idx])
models.eval_regression_log(reg, X.loc[te_idx], y.loc[te_idx])

## Estancia prolongada — clasificación

In [ ]:
yc = df['estancia_prolongada']
clf = models.train_classifier(X.loc[tr_idx], yc.loc[tr_idx], X.loc[te_idx], yc.loc[te_idx])
models.eval_classification(clf, X.loc[te_idx], yc.loc[te_idx])

## Explicación con SHAP

In [ ]:
import shap
exp, sv, Xs = models.shap_values(reg, X.loc[te_idx])
shap.plots.beeswarm(sv, max_display=15)

In [ ]:
shap.plots.scatter(sv[:, 'hora_llegada'], color=sv[:, 'ocupacion_llegada'])

## Modelo explicativo (todas las variables)

In [ ]:
Xe = models.prepare_X(df, features.FEATURES_EXPLICATIVO)
reg_e = models.train_regressor(Xe.loc[tr_idx], y.loc[tr_idx], Xe.loc[te_idx], y.loc[te_idx])
print(models.eval_regression_log(reg_e, Xe.loc[te_idx], y.loc[te_idx]))
_, sve, Xse = models.shap_values(reg_e, Xe.loc[te_idx])
shap.plots.bar(sve, max_display=15)

## Guardar para Power BI

In [ ]:
imp = models.shap_importance_table(sve, Xse)
imp.to_csv(config.DATA_PROCESSED / 'importancia_shap.csv', index=False)
df.loc[te_idx, 'los_pred_h'] = np.expm1(reg.predict(X.loc[te_idx]))
df.loc[te_idx, 'prob_prolongada'] = clf.predict_proba(X.loc[te_idx])[:, 1]
df.to_parquet(config.DATA_INTERIM / 'visitas_features.parquet', index=False)
import joblib; joblib.dump({'reg': reg, 'clf': clf, 'reg_explicativo': reg_e}, config.MODELS / 'modelos_los.joblib')